# ReviewAgent — Data Exploration

This notebook explores the datasets used in the ReviewAgent pipeline:
1. **MAALEJ labeled** — 5,008 human-annotated app reviews
2. **Synthetic** — 500 template-generated reviews
3. **RRGen** — 310,031 review-response pairs
4. **Combined training set** — MAALEJ + Synthetic (5,508 total)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = Path("../data/raw")

## 1. MAALEJ Labeled Dataset (5,008 reviews)

In [ ]:
maalej = json.loads((DATA_DIR / "maalej/maalej_labeled.json").read_text())
maalej_df = pd.DataFrame(maalej)

print(f"Total reviews: {len(maalej_df)}")
print(f"Columns: {list(maalej_df.columns)}")
print(f"\nSample reviews:")
maalej_df.head(3)

In [ ]:
# Label distribution
maalej_labels = Counter()
for r in maalej:
    for l in r["labels"]:
        maalej_labels[l] += 1

# Original label mapping
orig_labels = Counter(r["original_label"] for r in maalej)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mapped labels
labels, counts = zip(*sorted(maalej_labels.items(), key=lambda x: -x[1]))
colors = plt.cm.Set2(np.linspace(0, 1, len(labels)))
axes[0].barh(labels, counts, color=colors)
axes[0].set_xlabel("Count")
axes[0].set_title("MAALEJ — Mapped Labels")
for i, c in enumerate(counts):
    axes[0].text(c + 20, i, str(c), va='center')

# Original labels
orig_l, orig_c = zip(*sorted(orig_labels.items(), key=lambda x: -x[1]))
axes[1].barh(orig_l, orig_c, color=plt.cm.Pastel1(np.linspace(0, 1, len(orig_l))))
axes[1].set_xlabel("Count")
axes[1].set_title("MAALEJ — Original Labels")
for i, c in enumerate(orig_c):
    axes[1].text(c + 20, i, str(c), va='center')

plt.tight_layout()
plt.savefig("../notebooks/fig_maalej_labels.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Review text length distribution
maalej_df["text_len"] = maalej_df["text"].str.len()
maalej_df["word_count"] = maalej_df["text"].str.split().str.len()
maalej_df["primary_label"] = maalej_df["labels"].apply(lambda x: x[0] if x else "unknown")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(maalej_df["text_len"], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Character Length")
axes[0].set_ylabel("Count")
axes[0].set_title("MAALEJ — Review Length Distribution")
axes[0].axvline(maalej_df["text_len"].median(), color="red", linestyle="--", label=f'Median: {maalej_df["text_len"].median():.0f}')
axes[0].legend()

# Length per category
for label in sorted(maalej_df["primary_label"].unique()):
    subset = maalej_df[maalej_df["primary_label"] == label]
    axes[1].hist(subset["word_count"], bins=30, alpha=0.5, label=f'{label} ({len(subset)})')
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Count")
axes[1].set_title("MAALEJ — Word Count by Category")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("../notebooks/fig_maalej_lengths.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Text length stats:")
print(maalej_df["text_len"].describe().round(1))

## 2. Synthetic Data (500 reviews)

In [ ]:
synth = json.loads((DATA_DIR / "sample_reviews.json").read_text())
synth_df = pd.DataFrame(synth)

print(f"Total synthetic reviews: {len(synth_df)}")

# Label distribution
synth_labels = Counter()
for r in synth:
    for l in r["labels"]:
        synth_labels[l] += 1

# Rating distribution
synth_df["primary_label"] = synth_df["labels"].apply(lambda x: x[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Labels
labels, counts = zip(*sorted(synth_labels.items(), key=lambda x: -x[1]))
axes[0].barh(labels, counts, color=plt.cm.Set2(np.linspace(0, 1, len(labels))))
axes[0].set_xlabel("Count")
axes[0].set_title("Synthetic — Label Distribution")
for i, c in enumerate(counts):
    axes[0].text(c + 1, i, str(c), va='center')

# Rating per category
rating_by_label = synth_df.groupby("primary_label")["rating"].mean().sort_values()
rating_by_label.plot.barh(ax=axes[1], color="coral")
axes[1].set_xlabel("Average Rating")
axes[1].set_title("Synthetic — Avg Rating by Category")
axes[1].set_xlim(0, 5)

plt.tight_layout()
plt.savefig("../notebooks/fig_synthetic_dist.png", dpi=150, bbox_inches='tight')
plt.show()

## 3. Combined Training Set — MAALEJ + Synthetic

In [ ]:
# Combined label distribution (what the classifier trains on)
combined_labels = Counter()
for r in maalej + synth:
    for l in r["labels"]:
        combined_labels[l] += 1

labels_sorted = sorted(combined_labels.items(), key=lambda x: -x[1])
labels, counts = zip(*labels_sorted)
total = sum(counts)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(labels, counts, color=plt.cm.tab10(np.linspace(0, 1, len(labels))))
ax.set_xlabel("Count")
ax.set_title(f"Combined Training Set — Label Distribution (N={total})")

for i, (l, c) in enumerate(zip(labels, counts)):
    pct = c / total * 100
    ax.text(c + 20, i, f'{c} ({pct:.1f}%)', va='center')

# Highlight synthetic-only categories
for i, l in enumerate(labels):
    if l in ["performance", "compatibility"]:
        bars[i].set_edgecolor("red")
        bars[i].set_linewidth(2)

ax.annotate("Red border = synthetic only\n(no real labeled data)", 
            xy=(0.95, 0.95), xycoords="axes fraction", ha="right", va="top",
            fontsize=10, color="red",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"))

plt.tight_layout()
plt.savefig("../notebooks/fig_combined_labels.png", dpi=150, bbox_inches='tight')
plt.show()

## 4. RRGen Dataset (310K review-response pairs)

In [ ]:
rrgen = json.loads((DATA_DIR / "rrgen/rrgen_reviews.json").read_text())
print(f"Total RRGen reviews: {len(rrgen):,}")
print(f"Keys: {list(rrgen[0].keys())}")

# Rating distribution
ratings = [r.get("rating", 0) for r in rrgen if r.get("rating")]
rating_counts = Counter(ratings)

# Response length
resp_lens = [len(r.get("response", "")) for r in rrgen if r.get("response")]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating distribution
r_labels = sorted(rating_counts.keys())
r_counts = [rating_counts[r] for r in r_labels]
axes[0].bar(r_labels, r_counts, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Star Rating")
axes[0].set_ylabel("Count")
axes[0].set_title(f"RRGen — Rating Distribution (N={len(ratings):,})")

# Response length
axes[1].hist(resp_lens, bins=50, color="coral", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Response Length (chars)")
axes[1].set_ylabel("Count")
axes[1].set_title("RRGen — Developer Response Length")
axes[1].axvline(np.median(resp_lens), color="red", linestyle="--", label=f'Median: {np.median(resp_lens):.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig("../notebooks/fig_rrgen_overview.png", dpi=150, bbox_inches='tight')
plt.show()

# Unique apps
apps = set(r.get("app_id", "") for r in rrgen)
print(f"\nUnique apps: {len(apps)}")
print(f"Reviews with responses: {sum(1 for r in rrgen if r.get('response')):,}")
print(f"Median response length: {np.median(resp_lens):.0f} chars")

## 5. Data Summary Table

In [ ]:
summary = pd.DataFrame({
    "Dataset": ["MAALEJ labeled", "Synthetic", "Combined (training)", "RRGen (RAG corpus)"],
    "Reviews": [5008, 500, 5508, 310031],
    "Has Labels": ["Yes (human)", "Yes (template)", "Yes", "No"],
    "Has Responses": ["No", "No", "No", "Yes (310K)"],
    "Categories Covered": [
        "bug, feature, usability, praise, other",
        "bug, feature, usability, praise, performance, compatibility",
        "All 7",
        "N/A (unlabeled)",
    ],
    "Purpose": [
        "Classifier training (primary)",
        "Fill performance + compatibility gap",
        "Stage 1 classifier input",
        "Stage 4b RAG corpus",
    ],
})
summary.style.set_properties(**{'text-align': 'left'}).hide(axis='index')